# L5 · Lab B — Token-Level Rewards & Advantages

**RL for LLMs / RLHF — Vizuara AI Labs · Dr. Rajat Dandekar**

This is the **code version of the worked example** from the lecture (slides 50–52). PPO updates *every token* by its own advantage, so we must turn one end-of-sequence reward-model score into a **per-token** signal. You will compute, on a real generated sentence:

`R_t = −β·log(π_θ/π_ref)` per token, plus the reward-model score on the **last** token → reward-to-go `G_t` → advantage `A_t = G_t − baseline`.

> Runtime: **GPU**. ~3 minutes.

In [ ]:
!pip -q install "transformers==4.45.2" "datasets==3.0.1" >/dev/null

In [ ]:
# ✅ PROVIDED — get an HH reward model (load Lab A's ./rm, else quick-train)
import os, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from datasets import load_dataset
device = "cuda" if torch.cuda.is_available() else "cpu"
rm_tok = AutoTokenizer.from_pretrained("gpt2"); rm_tok.pad_token = rm_tok.eos_token; rm_tok.truncation_side = "left"
if os.path.isdir("rm"):
    rm = AutoModelForSequenceClassification.from_pretrained("rm").to(device).eval()
else:
    rm = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=1)
    rm.config.pad_token_id = rm_tok.eos_token_id; rm.to(device)
    ds = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0).select(range(4000))
    def _sc(t):
        e = rm_tok(t, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        return rm(**e).logits.squeeze(-1)
    opt = torch.optim.AdamW(rm.parameters(), lr=2e-5)
    for i in range(0, 4000, 8):
        rc, rr = _sc(ds["chosen"][i:i+8]), _sc(ds["rejected"][i:i+8])
        L = -F.logsigmoid(rc-rr).mean(); opt.zero_grad(); L.backward(); opt.step()
    rm.eval()
@torch.no_grad()
def rm_score(text):
    e = rm_tok(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    return rm(**e).logits.squeeze().item()

## 1 · Policy and reference

At the **start** of RLHF the policy equals the frozen reference, so the per-token KL is exactly 0. To make the KL toll visible we nudge the policy slightly — simulating a policy that has **drifted** a little during training.

In [ ]:
# ✅ PROVIDED — policy (slightly drifted) + frozen reference
tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token
policy = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
ref    = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
torch.manual_seed(0)
with torch.no_grad():
    for p in policy.parameters():
        if p.dim() >= 2: p.add_(0.02 * p.std() * torch.randn_like(p))   # small drift
print("policy + frozen reference ready")

In [ ]:
# ✅ PROVIDED — generate a response to a real HH prompt
BETA, GAMMA = 0.2, 1.0
data = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=3).select(range(5))
cut = data["chosen"][0].rfind("Assistant:"); prompt = data["chosen"][0][:cut+len("Assistant:")]
pids = tok(prompt, return_tensors="pt", truncation=True, max_length=128).input_ids.to(device)
with torch.no_grad():
    gen = policy.generate(pids, max_new_tokens=20, do_sample=True, top_p=0.9, top_k=0, pad_token_id=tok.eos_token_id)
resp = gen[0, pids.shape[1]:]; N = resp.shape[0]

def token_logprobs(model):
    logits = model(gen).logits
    logp = F.log_softmax(logits[:, :-1, :], dim=-1)
    return logp.gather(-1, gen[:, 1:].unsqueeze(-1)).squeeze(-1)[0][-N:]
with torch.no_grad():
    lp_pi  = token_logprobs(policy)
    lp_ref = token_logprobs(ref)
RM = rm_score(tok.decode(gen[0]))
print("response:", tok.decode(resp)); print("reward-model score (terminal):", round(RM, 3))

## 2 · 📝 TODO — per-token rewards, returns, advantages

Fill in the three method-defining computations (`lp_pi`, `lp_ref`, `RM`, `N`, `BETA`, `GAMMA` are ready).

In [ ]:
# 📝 TODO 1 — per-token reward.  KL_t = log(pi/pi_ref); reward = -beta*KL; add RM on the LAST token.
kl = None            # lp_pi - lp_ref           (shape [N])
R  = None            # -BETA * kl  (a clone), then R[-1] = R[-1] + RM

# 📝 TODO 2 — returns (reward-to-go), gamma=1:  G_t = R_t + gamma * G_{t+1}
G = torch.zeros(N, device=device)
# for t in reversed(range(N)): ...

# 📝 TODO 3 — advantage with the simplest baseline (the mean return; Lab D learns a per-token V):
baseline = None      # G.mean()
A = None             # G - baseline

raise NotImplementedError("implement TODO 1-3, then delete this line")

## 3 · The per-token table

This reproduces the lecture's worked-example table, but with **real model numbers**.

In [ ]:
# ✅ PROVIDED — print the per-token breakdown
dec = [tok.decode([t]) for t in resp.tolist()]
print(f"{'token':>10} {'logπθ':>8} {'logπref':>8} {'KL':>7} {'R_t':>7} {'G_t':>7} {'A_t':>7}")
for t in range(N):
    print(f"{dec[t]!r:>10} {lp_pi[t]:8.2f} {lp_ref[t]:8.2f} {kl[t]:7.2f} {R[t]:7.3f} {G[t]:7.3f} {A[t]:7.3f}")
print(f"\nmean |KL| per token: {kl.abs().mean():.3f}   (small — the leash is a gentle nudge)")
print(f"baseline (mean return): {baseline:.3f}")

## 4 · Questions to answer

1. Most tokens have a tiny KL toll while the **last** token carries a big jump — why? Which term causes each?
2. With γ=1, what does `G_t` equal for the very first token, in words?
3. A token with `A_t > 0` gets pushed **up** by PPO. Point to one such token in your table and say (intuitively) why it was "better than the baseline expected."
4. Change `drift` from 0.02 to 0.2 and re-run. What happens to the KL column and to the readability of the response? (This is exactly what the KL leash prevents.)
